# Fraud Detection System
## 06 — Deployment

| | |
|---|---|
| **Author** | Jose Lopez Pino |
| **Framework** | CRISP-DM + Lean |
| **Phase** | 6 — Deployment |
| **Project** | fraud-detection · applied-data-science-portfolio |
| **Dataset** | Credit Card Fraud Detection — ULB Machine Learning Group (Kaggle) |
| **Date** | 2026-04 |

---

> **Executive Summary:**
> The Fraud Detection System is deployment-ready. The winning model (Random Forest +
> class_weight) at cost-optimized threshold t*=0.0414 catches 81.7% of fraud (58/71 cases)
> with an Expected Loss of USD 4,320 — a 79.7% reduction vs the naive baseline (USD 21,300).
> This notebook documents the Model Card, MLOps checklist, executive summary, and Streamlit
> app specification. The CRISP-DM cycle is complete.

## Table of Contents

1. [Model Card](#1-model-card)
2. [Model Performance — Final Table](#2-model-performance--final-table)
3. [MLOps Checklist](#3-mlops-checklist)
4. [Executive Summary — Business Audience](#4-executive-summary--business-audience)
5. [Business Recommendations](#5-business-recommendations)
6. [Limitations](#6-limitations)
7. [Streamlit App — Specification](#7-streamlit-app--specification)
8. [Project Retrospective — CRISP-DM Cycle Complete](#8-project-retrospective--crispDM-cycle-complete)
9. [LEAN Filter — Waste Elimination Review](#9-lean-filter--waste-elimination-review)
10. [Decisions Log](#10-decisions-log)

---

**← Previous:** [05 — Evaluation](./05_evaluation.ipynb)

---
## 1. Model Card

| Field | Details |
|---|---|
| **Model type** | Random Forest Classifier |
| **Imbalance strategy** | class_weight='balanced' |
| **Task** | Binary classification — fraud vs legitimate |
| **Training data** | ULB Credit Card Fraud dataset — 284,807 transactions (2013, European cardholders) |
| **Train set size** | ~199,000 rows (70% stratified split) |
| **Features used** | 30 features: Time, Amount (scaled), V1–V28 (PCA) |
| **Target variable** | Class — 1 = Fraud, 0 = Legitimate |
| **Framework** | Scikit-learn 1.8.0 |
| **Training date** | April 2026 |
| **Model artifact** | `models/model_v1_2026-04.pkl` |
| **Classification threshold** | t* = 0.0414 (cost-optimized, not default 0.5) |
| **Cost ratio** | 20x — missing fraud costs 20x more than blocking a legitimate transaction |

**Key design decision:** The threshold was NOT left at the default 0.5.
It was calibrated by minimizing `EL(t) = FN(t) × $300 + FP(t) × $15`
on the validation set. This is the core differentiator of this system.

---
## 2. Model Performance — Final Table

### 2.1 Classification metrics (test set, threshold = 0.0414)

| Metric | Value | Target | Met? |
|---|---|---|---|
| **AUC-PR** (primary) | 0.7861 | ≥ 0.85 | ❌ |
| **Recall** (fraud class) | 0.8169 | ≥ 0.80 | ✅ |
| **Precision** (fraud class) | 0.6744 | ≥ 0.70 | ❌ |
| **F1** (fraud class) | 0.7389 | — | — |
| **AUC-ROC** | — | — | — |

### 2.2 Business metrics (test set)

| Metric | Value |
|---|---|
| Fraud cases in test set | 71 |
| Fraud caught (True Positives) | 58 (81.7%) |
| Fraud missed (False Negatives) | 13 |
| Legitimate blocked (False Positives) | 28 |
| Expected Loss at t* | USD 4,320 |
| Naive baseline (predict all legitimate) | USD 21,300 |
| **Savings vs naive baseline** | **USD 16,980 (79.7% reduction)** |

### 2.3 Honest assessment of unmet targets

**AUC-PR = 0.7861 (target ≥ 0.85):**
AUC-PR measures ranking quality across all thresholds — the model ranks fraud cases
reasonably well but does not reach the target. This reflects the fundamental difficulty
of the anonymized PCA features. Improvement paths: XGBoost with hyperparameter tuning,
feature interactions, or ensemble stacking.

**Precision = 0.6744 (target ≥ 0.70):**
The aggressive threshold (t*=0.0414) driven by the 20x cost ratio produces more false
positives. This is the expected precision-recall tradeoff — the business explicitly
decided that missing fraud is 20x more costly than blocking a legitimate transaction.
At a lower cost ratio (e.g. 5x), precision would improve at the expense of recall.

---
## 3. MLOps Checklist

### Reproducibility
- [x] Random seed fixed (`random_state=42` in all models and splits)
- [x] Requirements pinned (`requirements.txt` with exact versions)
- [x] Data versioned (Kaggle CLI command documented in notebook 02)
- [x] Model saved (`joblib` — `models/model_v1_2026-04.pkl`)

### Model Versioning
- [x] Model artifact saved to `models/` folder
- [x] Model filename follows convention: `model_v1_2026-04.pkl`
- [x] Training parameters logged in Decisions Log (notebooks 03–05)
- [x] Model Card documented (section 1 above)

### Monitoring (awareness level)
- [x] Data drift noted — dataset covers 2 days in 2013; fraud patterns evolve over time
- [x] Model limitations documented (section 6 below)
- [x] Retraining triggers defined — performance degradation > 5% in AUC-PR or recall

### Pipeline
- [x] Full pipeline reproducible from raw data: nb02 → nb03 → nb04 → nb05
- [x] No absolute paths in any notebook
- [x] Test set used only once (notebook 05)

---
## 4. Executive Summary — Business Audience

---

### The Problem

Financial institutions face two simultaneous failure modes in fraud detection:
fraud that passes undetected (direct monetary loss) and legitimate transactions
that get blocked (customer friction, churn risk). Standard ML approaches that
optimize for accuracy fail silently — a model that labels every transaction as
legitimate achieves 99.8% accuracy while catching zero fraud.

### The Approach

This system reframes fraud detection as a **cost minimization problem**, not a
classification problem. The model is calibrated to minimize total expected loss:

> `Expected Loss = (missed fraud × $300) + (blocked legitimate × $15)`

The classification threshold is set at **t* = 0.0414** — not the arbitrary default
of 0.5 — because missing one fraud is 20x more costly than a false alarm.

### The Results

On a holdout test set of 42,721 transactions (71 fraudulent):

| | Naive approach | This system |
|---|---|---|
| Fraud caught | 0 of 71 (0%) | 58 of 71 (81.7%) |
| Legitimate blocked | 0 | 28 |
| Expected Loss | USD 21,300 | USD 4,320 |
| **Savings** | — | **USD 16,980 (79.7%)** |

### The Recommendation

Deploy this system as a **first-line fraud filter** with human review for flagged
transactions. The cost ratio parameter (currently 20x) should be calibrated
against actual loss data from the institution's fraud records for optimal performance.

---
## 5. Business Recommendations

| Priority | Action | Expected Impact |
|---|---|---|
| 🔴 HIGH | Calibrate `cost_fn` and `cost_fp` with real institutional loss data | Threshold t* will shift to reflect actual costs — improves both precision and recall |
| 🔴 HIGH | Implement human review queue for flagged transactions (28 FP + 58 TP) | Recovers false positives before customer impact — reduces friction |
| 🟡 MEDIUM | Retrain monthly with new transaction data | Fraud patterns evolve — model trained on 2013 data will degrade over time |
| 🟡 MEDIUM | Test XGBoost with hyperparameter tuning to close AUC-PR gap (0.7861 → 0.85) | May recover unmet AUC-PR target without sacrificing recall |
| 🟢 LOW | Add real-time scoring via REST API wrapper around saved model | Enables integration with transaction processing systems |

---
## 6. Limitations

**Data limitations:**
- Dataset covers only 2 days of transactions (September 2013, European cardholders)
- Fraud patterns are time-dependent — model may not generalize to current fraud tactics
- Features V1–V28 are anonymized PCA components — no domain feature engineering possible
- COST_FN and COST_FP are assumed values — not derived from real institutional loss data

**Model limitations:**
- AUC-PR target (0.85) not met — current model achieves 0.7861
- Precision target (0.70) not met — current model achieves 0.6744 at t*=0.0414
- Random Forest is not natively interpretable at transaction level
- No concept drift monitoring implemented

**Deployment limitations:**
- Model not tested on real-time streaming data
- No A/B testing framework defined
- Retraining pipeline not automated

---
## 7. Streamlit App — Specification

The `apps/fraud-detection-app/` Streamlit app exposes the cost framework
to business stakeholders without requiring code interaction.

### Core features

| Feature | Description |
|---|---|
| **cost_fn slider** | USD 50 – 1,000 — cost of missing one fraud |
| **cost_fp slider** | USD 5 – 100 — cost of blocking one legitimate transaction |
| **cost_ratio display** | Computed automatically from sliders |
| **Optimal threshold** | Recomputed live from new cost inputs |
| **Expected Loss chart** | EL curve with t* marker — updates with sliders |
| **Results table** | Recall, Precision, F1, EL at new t* |
| **Savings vs naive** | USD savings and % reduction — key business output |

### App structure

```
apps/fraud-detection-app/
├── app.py                  ← main Streamlit app
├── requirements.txt        ← streamlit, joblib, scikit-learn, pandas, plotly
└── model/
    └── model_v1_2026-04.pkl  ← copy of trained model
```

> The app will be built in the next sprint after README completion.

---
## 8. Project Retrospective — CRISP-DM Cycle Complete

### What worked well

- **Expected Loss Framework** — reframing fraud detection as cost minimization
  was the right lens. It produced a business-interpretable result (USD savings)
  that goes beyond standard ML metrics.
- **Lean filter at every phase** — prevented exhaustive EDA of all 28 PCA features,
  unnecessary hyperparameter tuning before model selection, and ADASYN overhead.
- **Stratified split** — preserved fraud rate across all splits. Critical with 0.17% fraud rate.
- **Isolation Forest benchmark** — confirmed that labeled data adds significant value
  over unsupervised anomaly detection for this dataset.

### What to improve in the next cycle

- **AUC-PR gap** — 0.7861 vs target 0.85. Next step: XGBoost with grid search
  on `n_estimators`, `max_depth`, `learning_rate`.
- **Cost calibration** — COST_FN and COST_FP were assumed. Real institutional
  loss data would produce a more accurate threshold.
- **Feature engineering** — V1–V28 anonymization limits domain features.
  Amount × Time interaction terms could be explored.

### CRISP-DM phases completed

| Phase | Notebook | Key Output | Status |
|---|---|---|---|
| 1 — Business Understanding | 01 | Problem Statement Canvas + Expected Loss Framework | ✅ |
| 2 — Data Understanding | 02 | EDA + discriminative feature ranking | ✅ |
| 3 — Data Preparation | 03 | Scaled splits + 3 imbalance strategies | ✅ |
| 4 — Modeling | 04 | 9 configurations compared — RF class_weight wins | ✅ |
| 5 — Evaluation | 05 | t*=0.0414, EL=USD 4,320, savings=79.7% | ✅ |
| 6 — Deployment | 06 | Model Card + MLOps + Executive Summary + App spec | ✅ |

---
## 9. LEAN Filter — Waste Elimination Review

| LEAN Question | Answer | Action |
|---|---|---|
| Does the executive summary use plain language? | ✅ Yes — no technical jargon, USD figures front and center | Proceed |
| Are unmet targets documented honestly? | ✅ Yes — AUC-PR and Precision gaps explained with root cause | Proceed |
| Is the Streamlit app over-specified? | ✅ No — only features that serve a business decision are included | Proceed |
| Does the retrospective lead to actionable next steps? | ✅ Yes — XGBoost tuning and cost calibration are concrete next actions | Proceed |

---
## 10. Decisions Log

| # | Decision | Rationale | Alternatives Considered | LEAN Value? |
|---|---|---|---|---|
| 1 | Report unmet targets honestly (AUC-PR, Precision) | Intellectual honesty is non-negotiable — a portfolio that hides failures is a red flag | Adjust targets post-hoc | ✅ |
| 2 | Explain precision gap as expected cost-ratio tradeoff | Demonstrates understanding of precision-recall tradeoff — not a model failure but a business decision | Flag as bug | ✅ |
| 3 | Streamlit app specified but not built in this notebook | Separation of concerns — model + app are distinct deliverables | Build app inline | ✅ |
| 4 | Recommend real cost calibration as first priority | COST_FN/FP assumptions drive all results — real data would improve every metric | Keep assumed costs | ✅ |